# S-Fig 1 — K-Aggregation

AUROC vs K at representative context lengths.  
**Source**: analysis.csv (all k values)  
**Tasks**: main + OSA (depression and CVD excluded)  
**Layout**: 3 rows × 2 cols. Increase ROW_H for larger panels.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
# Tasks: main 5 + OSA (exclude depression and CVD)
TASKS    = MAIN_TASKS + ["osa_binary_apples_postqc"]
CONTEXTS = ["40m", "120m", "240m"]   # context lengths shown per panel
HEADS    = ["lstm", "transformer"]
N_COLS   = 2
N_ROWS   = 3   # 6 tasks → 3 × 2, all rows full
ROW_H    = 2.4   # inches per row — increase for larger panels

import pandas as pd
_ana_path = WORKSPACE_ROOT / "final_results" / "phase0_v3" / "collected" / "analysis.csv"
df_all_k = pd.read_csv(_ana_path)
df_all_k["context_length_min"] = df_all_k["context_length"].map(
    lambda s: {"30s": 0.5, "10m": 10.0, "40m": 40.0,
               "80m": 80.0, "120m": 120.0, "240m": 240.0}.get(str(s).strip())
)
print("Tasks:", TASKS)
print("Loaded:", df_all_k.shape, "rows")

In [ ]:
labels = [chr(97 + i) for i in range(len(TASKS))]

fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(FULL_W, N_ROWS * ROW_H))
axes_flat = axes.flatten()

for i, (ax, task) in enumerate(zip(axes_flat, TASKS)):
    panels.k_agg_panel(ax, df_all_k, task, heads=HEADS, contexts=CONTEXTS)
    ax.set_title(TASK_LABEL.get(task, task), fontsize=8)
    add_panel_label(ax, f"({labels[i]})")

fig.tight_layout(h_pad=0.8, w_pad=1.0)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "sfig1_k_aggregation")
print("Saved →", FINAL_OUT / "sfig1_k_aggregation.pdf")